In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

In [2]:
files = [
    'two_class_1s_no.csv',
    'two_class_1s_0.5.csv',
    'two_class_1s_0.8.csv',
    'two_class_2s_no.csv',
    'two_class_2s_0.5.csv',
    'two_class_2s_0.8.csv',
    'two_class_3s_no.csv',
    'two_class_3s_0.5.csv',
    'two_class_3s_0.8.csv',
    'two_class_4s_no.csv',
    'two_class_4s_0.5.csv',
    'two_class_4s_0.8.csv',
    'two_class_5s_no.csv',
    'two_class_5s_0.5.csv',
    'two_class_5s_0.8.csv'
]

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/extracted_features_data/2_class/train_set/'

In [3]:
def screening_cross_val(X, y, groups, model_type, random_state=42):
    """
    Compare different sampling techniques for imbalanced classification
    """
    
    # Define sampling strategies to test
    sampling_strategies = {
        'none': None,
        'oversample_smote': SMOTE(random_state=random_state),
        'undersample_tomek_links': TomekLinks(),
        'combined_smote_tomek': SMOTETomek(random_state=random_state)
    }
    
    results = {}
    conf_mat_data = {}
    
    for strategy_name, sampler in sampling_strategies.items():
        print(f"\nTesting {strategy_name}...")
        
        # Cross-validation setup
        # gkf = GroupKFold(n_splits=5)  # Using 5 folds for faster initial testing
        gkf = StratifiedGroupKFold(n_splits=5)  # Stratified Group K-Fold for better class balance
        fold_results = []
        
        # For confusion matrix
        predicted_targets = np.array([])
        actual_targets = np.array([])
        
        # split data
        for fold_num, (train_idx_fold, test_idx_fold) in enumerate(gkf.split(X, y, groups)):
            X_train_fold, X_test_fold = X.iloc[train_idx_fold], X.iloc[test_idx_fold]
            y_train_fold, y_test_fold = y.iloc[train_idx_fold], y.iloc[test_idx_fold]

            
            # Apply sampling (if any)
            if sampler is not None:
                try:
                    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_fold, y_train_fold)
                    X_train_resampled = pd.DataFrame(X_train_resampled, columns=X_train_fold.columns)
                    y_train_resampled = pd.Series(y_train_resampled)
                except Exception as e:
                    print(f"Sampling failed for {strategy_name}: {e}")
                    continue
            else:
                X_train_resampled = X_train_fold
                y_train_resampled = y_train_fold
            
            if model_type == 'xgb':
                # Use XGBoost with DEFAULT parameters
                model = XGBClassifier(
                    random_state=random_state,
                    eval_metric='logloss',  # Suppress warning
                )
            elif model_type == 'dt':
                # Use Decision Tree with DEFAULT parameters
                model = DecisionTreeClassifier(
                    random_state=random_state,
                )
            elif model_type == 'rf':
                # Use RandomForest with DEFAULT parameters
                model = RandomForestClassifier(
                    random_state=random_state,
            )
            
            # Encode labels
            label_encoder = LabelEncoder()
            y_train_encoded = label_encoder.fit_transform(y_train_resampled)
            
            # Fit model
            model.fit(X_train_resampled, y_train_encoded)
            
            # Predict
            predictions = label_encoder.inverse_transform(model.predict(X_test_fold))
            
            # For confusion matrix
            predicted_targets = np.append(predicted_targets, predictions)
            actual_targets = np.append(actual_targets, y_test_fold)
            # Store the confusion matrix data
            conf_mat_data[strategy_name] = {
                'predicted': predicted_targets,
                'actual': actual_targets
            }
            
            # Calculate met# split datarics
            accuracy = accuracy_score(y_test_fold, predictions)
            report_dict = classification_report(y_test_fold, predictions, output_dict=True)

            precision_void = report_dict.get("void", {}).get("precision", 0.0)
            recall_void = report_dict.get("void", {}).get("recall", 0.0)
            f1_void = report_dict.get("void", {}).get("f1-score", 0.0)

            precision_non_void = report_dict.get("non-void", {}).get("precision", 0.0)
            recall_non_void = report_dict.get("non-void", {}).get("recall", 0.0)
            f1_non_void = report_dict.get("non-void", {}).get("f1-score", 0.0)

            macro_f1 = report_dict.get("macro avg", {}).get("f1-score", 0.0)
            weighted_f1 = report_dict.get("weighted avg", {}).get("f1-score", 0.0)
            
            # Store detailed results
            fold_results.append({
                'fold': fold_num,
                'recall_void': recall_void,
                'precision_# split datavoid': precision_void,
                'f1_void': f1_void,
                'macro_f1': macro_f1,
                'weighted_f1': weighted_f1,
                'accuracy': accuracy,
                'precision_non_void': precision_non_void,  # Assuming 'non-void' is majority
                'recall_non_void': recall_non_void,
                'f1_non_void': f1_non_void,
                'class_distribution_train': dict(y_train_resampled.value_counts()),
                'class_distribution_test': dict(y_test_fold.value_counts())
            })
            

        # Calculate summary statistics
        if fold_results:  # Only if we have valid results
            results[strategy_name] = {
                'mean_accuracy': np.mean([f['accuracy'] for f in fold_results]),
                'std_accuracy': np.std([f['accuracy'] for f in fold_results]),
                'mean_recall_minority': np.mean([f['recall_void'] for f in fold_results]),
                'std_recall_minority': np.std([f['recall_void'] for f in fold_results]),
                'mean_f1_minority': np.mean([f['f1_void'] for f in fold_results]),
                'std_f1_minority': np.std([f['f1_void'] for f in fold_results]),
                'mean_f1_majority': np.mean([f['f1_non_void'] for f in fold_results]),
                'std_f1_majority': np.std([f['f1_non_void'] for f in fold_results]),
                'macro_f1': np.mean([f['macro_f1'] for f in fold_results]),
                'std_macro_f1': np.std([f['macro_f1'] for f in fold_results]),
                'weighted_f1': np.mean([f['weighted_f1'] for f in fold_results]),
                'std_weighted_f1': np.std([f['weighted_f1'] for f in fold_results]),
                'fold_details': fold_results
            }
    
    return results, conf_mat_data

## Decision Tree

In [4]:
file_results_dt = {}
conf_mat_data = {}

for file in tqdm(files, desc="Producing results for different sampling techniques - train data only"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    file_results_dt[exp_name], conf_mat_data[exp_name] = screening_cross_val(X, y, groups,'dt', 42)
    
# pickle the reults
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/dt_cv_results_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(file_results_dt, f)
# pickle the confusion matrix data
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/dt_conf_mat_data_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(conf_mat_data, f)   

Producing results for different sampling techniques - train data only:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - train data only:   7%|▋         | 1/15 [00:05<01:10,  5.04s/it]

Analysing 1s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - train data only:  13%|█▎        | 2/15 [00:16<01:55,  8.89s/it]

...

Producing results for different sampling techniques - train data only:  87%|████████▋ | 13/15 [01:30<00:06,  3.44s/it]

Analysing 5s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - train data only:  93%|█████████▎| 14/15 [01:32<00:02,  2.87s/it]

Analysing 5s_0.8

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - train data only: 100%|██████████| 15/15 [01:35<00:00,  6.40s/it]


## Random Forest

In [5]:
file_results_rf = {}
conf_mat_data = {}
for file in tqdm(files, desc="Producing results for different sampling techniques - working data only"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    file_results_rf[exp_name], conf_mat_data[exp_name] = screening_cross_val(X, y, groups,'rf', 42)
    
# pickle the results
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/rf_cv_results_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(file_results_rf, f)
    
# pickle the confusion matrix data
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/rf_conf_mat_data_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(conf_mat_data, f)

Producing results for different sampling techniques - working data only:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only:   7%|▋         | 1/15 [00:23<05:26, 23.34s/it]

Analysing 1s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only:  13%|█▎        | 2/15 [01:14<08:38, 39.85s/it]

...

Producing results for different sampling techniques - working data only:  87%|████████▋ | 13/15 [06:33<00:29, 14.81s/it]

Analysing 5s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only:  93%|█████████▎| 14/15 [06:40<00:12, 12.43s/it]

Analysing 5s_0.8

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only: 100%|██████████| 15/15 [06:57<00:00, 27.82s/it]


## XGBoost

In [6]:
file_results_xgb = {}
conf_mat_data = {}
for file in tqdm(files, desc="Producing results for different sampling techniques - working data only"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}" 
    print(f"Analysing {exp_name}")
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    file_results_xgb[exp_name], conf_mat_data[exp_name] = screening_cross_val(X, y, groups,'xgb', 42)
    
# pickle the results
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/xgb_cv_results_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(file_results_xgb, f)
# pickle the confusion matrix data
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/xgb_conf_mat_data_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(conf_mat_data, f)   

Producing results for different sampling techniques - working data only:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only:   7%|▋         | 1/15 [00:13<03:03, 13.10s/it]

Analysing 1s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only:  13%|█▎        | 2/15 [00:31<03:29, 16.10s/it]

...

Producing results for different sampling techniques - working data only:  87%|████████▋ | 13/15 [02:39<00:17,  8.70s/it]

Analysing 5s_0.5

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only:  93%|█████████▎| 14/15 [02:46<00:08,  8.14s/it]

Analysing 5s_0.8

Testing none...

Testing oversample_smote...

Testing undersample_tomek_links...

Testing combined_smote_tomek...


Producing results for different sampling techniques - working data only: 100%|██████████| 15/15 [02:56<00:00, 11.76s/it]
